In [2]:
!pip install -U bitsandbytes>=0.46.1

In [5]:
import pandas as pd
df=pd.read_parquet("/root/autodl-fs/Manchu_OCR/test.parquet")
df.to_csv("/root/test.csv")

In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
#============TRAINING============#
import os
#Add this line if you cannot connect Hugging Face in China
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
import torch
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from transformers import (
    Qwen3VLForConditionalGeneration,
    AutoProcessor,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, TaskType
from torch.utils.data import Dataset
from qwen_vl_utils import process_vision_info

CONFIG = {
    'model_id': "Qwen/Qwen3-VL-4B-Instruct",
    'data_parquet': "/root/autodl-fs/Manchu_OCR/train.parquet",#your own path of train.parquet
    'image_dir': "/root/autodl-fs/Manchu_OCR/train",#your own path of train images
    'output_dir': "/root/autodl-tmp/qwen_lora_output_4b_qlora",#your own savepath of LoRA weight
    'epochs': 9,
    'batch_size': 64,         
    'gradient_accum': 1,       
    'learning_rate': 2e-4,
    'min_pixels': 32 * 32,
    'max_pixels': 384 * 384,    
    
    'dataloader_num_workers': 16,          
    'dataloader_pin_memory': True,
    'dataloader_persistent_workers': True,
    'dataloader_prefetch_factor': 4,
}

class ManchuWordDataset(Dataset):
    def __init__(self, df, processor, image_dir):
        self.df = df
        self.processor = processor
        self.image_dir = image_dir
        self.ignore_index = -100
        self.prompt_text = "识别满文单词：" #Prompt to Recognize Manchu word
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = os.path.join(self.image_dir, row['filename'])
        target_text = row['roman']
      
        image = Image.open(image_path).convert("RGB")
      
        messages = [
            {"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": self.prompt_text}]},
            {"role": "assistant", "content": target_text}
        ]
        text_prompt = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        image_inputs, video_inputs = process_vision_info(messages)
       
        inputs = self.processor(text=[text_prompt], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt")
      
        for k in ["input_ids", "attention_mask"]:
            if k in inputs: inputs[k] = inputs[k].squeeze(0)
        if "mm_token_type_ids" in inputs:
            inputs["mm_token_type_ids"] = inputs["mm_token_type_ids"].squeeze(0)
        input_ids = inputs["input_ids"]
        labels = input_ids.clone()
        prompt_only_messages = [messages[0]]
        prompt_only_text = self.processor.apply_chat_template(prompt_only_messages, tokenize=False, add_generation_prompt=True)
        prompt_only_inputs = self.processor(text=[prompt_only_text], images=image_inputs, videos=video_inputs, return_tensors="pt")
        prompt_length = prompt_only_inputs["input_ids"].squeeze(0).shape[0]
      
        labels[:prompt_length] = self.ignore_index
        inputs["labels"] = labels
        return inputs

def collate_fn(batch):
    input_ids = [item["input_ids"] for item in batch]
    labels = [item["labels"] for item in batch]
    batch_dict = {
        "input_ids": torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=0),
        "labels": torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=-100),
    }
    batch_dict["attention_mask"] = batch_dict["input_ids"].ne(0).long()
  
    if "pixel_values" in batch[0] and batch[0]["pixel_values"] is not None:
        batch_dict["pixel_values"] = torch.cat([item["pixel_values"] for item in batch], dim=0)
    if "image_grid_thw" in batch[0] and batch[0]["image_grid_thw"] is not None:
        batch_dict["image_grid_thw"] = torch.cat([item["image_grid_thw"] for item in batch], dim=0)
    if "mm_token_type_ids" in batch[0] and batch[0]["mm_token_type_ids"] is not None:
        mm = [item["mm_token_type_ids"] for item in batch]
        batch_dict["mm_token_type_ids"] = torch.nn.utils.rnn.pad_sequence(mm, batch_first=True, padding_value=0)
    return batch_dict

# ===========================
# 主流程
# ===========================
def main():

  
    processor = AutoProcessor.from_pretrained(CONFIG['model_id'], min_pixels=CONFIG['min_pixels'], max_pixels=CONFIG['max_pixels'])
  
    df = pd.read_parquet(CONFIG['data_parquet'])
    train_df, eval_df = train_test_split(df, test_size=0.05, random_state=42)
    train_dataset = ManchuWordDataset(train_df, processor, CONFIG['image_dir'])
    eval_dataset = ManchuWordDataset(eval_df, processor, CONFIG['image_dir'])
    print(f"Dataset loaded: {len(train_dataset)} images ")
    
    print("\nLoading Model...")
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        CONFIG['model_id'],
        torch_dtype=torch.bfloat16,
        attn_implementation="flash_attention_2",
        device_map="auto",
        quantization_config=quantization_config,
    )
    model.gradient_checkpointing_enable()
    peft_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=64, lora_alpha=128, lora_dropout=0.05,
        target_modules="all-linear",
        modules_to_save=["embed_tokens", "lm_head"]
    )
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()
    
    # 显存监控（训练前打印）
    print(f"VRAM Occupancy: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
    print(f"Max VRAM Occupancy: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")
    
    print("\nStart Training...")
    training_args = TrainingArguments(
        output_dir=CONFIG['output_dir'],
        per_device_train_batch_size=CONFIG['batch_size'],
        gradient_accumulation_steps=CONFIG['gradient_accum'],
        learning_rate=CONFIG['learning_rate'],
        num_train_epochs=CONFIG['epochs'],
        bf16=True,
        logging_steps=50,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        save_total_limit=3,
        remove_unused_columns=False,
        report_to="none",
        
       
        dataloader_num_workers=CONFIG['dataloader_num_workers'],
        dataloader_pin_memory=CONFIG['dataloader_pin_memory'],
        dataloader_persistent_workers=CONFIG['dataloader_persistent_workers'],
        dataloader_prefetch_factor=CONFIG['dataloader_prefetch_factor'],
        
        optim="adamw_torch",
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=collate_fn,
    )
    trainer.train()
  
    final_save_path = os.path.join(CONFIG['output_dir'], "final_lora")
    trainer.model.save_pretrained(final_save_path)
    processor.save_pretrained(final_save_path)
    print(f"✓ Training complete. QLoRA weights saved in: {final_save_path}")

if __name__ == "__main__":
    main()


Qwen3-VL-4B QLoRA 加速版启动 (已极致吃满 3090 24GB)
batch=64 + accum=1 + max_pixels=384*384 → 预计峰值 21.5~23GB
数据集加载完成: 训练 57000 条

[3/4] 加载模型 + 4bit 量化 (QLoRA)...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

/root/miniconda3/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 938,999,808 || all params: 5,376,815,616 || trainable%: 17.4639
模型+LoRA加载后显存占用: 4.77 GB
峰值已分配显存: 4.77 GB

[4/4] 启动训练... (batch=64 + accum=1)


Casting fp32 inputs back to torch.bfloat16 for flash-attn compatibility.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Epoch,Training Loss,Validation Loss
1,0.141140,0.143486
2,0.061797,0.068172
3,0.031652,0.056033
4,0.016974,0.047505
5,0.005612,0.033921
6,0.002124,0.034582
7,0.000419,0.036077
8,0.000423,0.036261
9,0.000028,0.036519


✓ 训练完成！QLoRA 权重已保存至: /root/autodl-tmp/qwen_lora_output_4b_qlora/final_lora


In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
SYNTHETIC DATA (VALIDATION SET) TEST
"""
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
import torch
import pandas as pd
from PIL import Image
from tqdm import tqdm
import jiwer
from sklearn.model_selection import train_test_split
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from peft import PeftModel
from qwen_vl_utils import process_vision_info

# ===========================
# Paths and Configurations
# ===========================
CONFIG = {
    'base_model_id': "Qwen/Qwen3-VL-4B-Instruct",                    
    'lora_path': "Um1neko/Qwen3VL-Instruct-4B-ManchuOCR",  # PATH OF YOUR QLORA WEIGHT
    'data_parquet': "/root/autodl-fs/Manchu_OCR/train.parquet", # PATH OF YOUR TRAIN.PARQUET
    'image_dir': "/root/autodl-fs/Manchu_OCR/train", # PATH OF YOUR TRAIN 
   
    'min_pixels': 32 * 32,
    'max_pixels': 384 * 384,         
   
    'leakage_threshold': 5.0,
}

def main():
    print("\n" + "="*70)
    print("Starting Qwen3-VL-4B QLoRA Manchu OCR Evaluation (Unseen Words Mode)")
    print("="*70)
   
    # ===========================
    # 1. Data Splitting and Overlap Check
    # ===========================
    print("\n[1/4] Reproducing dataset split and checking overlap...")
    df = pd.read_parquet(CONFIG['data_parquet'])
   
    # Maintain the same random seed and split ratio as used in training
    train_df, eval_df = train_test_split(df, test_size=0.05, random_state=42)
   
    # Extract unique word labels from the training set
    train_vocab = set(train_df['roman'].unique())
   
    # Check overlap
    overlap_mask = eval_df['roman'].isin(train_vocab)
    overlap_count = overlap_mask.sum()
    total_eval = len(eval_df)
    overlap_percentage = (overlap_count / total_eval) * 100
   
    print(f" > Total validation samples: {total_eval}")
    print(f" > Samples seen in training set (leakage): {overlap_count}")
    print(f" > Label overlap rate: {overlap_percentage:.2f}%")
   
    if overlap_percentage > CONFIG['leakage_threshold']:
        print(f"\n[!] Warning: Overlap rate ({overlap_percentage:.2f}%) exceeds threshold ({CONFIG['leakage_threshold']}%).")
        print("[!] Filtering out images with seen labels...")
        eval_df = eval_df[~overlap_mask]
        print(f" > Filtering complete. Final unseen validation set size: {len(eval_df)}")
    else:
        print(f"\n[✓] Overlap rate ({overlap_percentage:.2f}%) is within the acceptable range. Evaluating the full validation set.")
   
    if len(eval_df) == 0:
        print("\n[x] Error: Validation set is empty after filtering. Evaluation aborted.")
        return
   
    # ===========================
    # 2. Load Processor and Model
    # ===========================
    print("\n[2/4] Loading processor...")
    processor = AutoProcessor.from_pretrained(
        CONFIG['lora_path'],
        min_pixels=CONFIG['min_pixels'],
        max_pixels=CONFIG['max_pixels']
    )
   
    print("\n[3/4] Loading base model and integrating QLoRA adapter...")
    base_model = Qwen3VLForConditionalGeneration.from_pretrained(
        CONFIG['base_model_id'],
        torch_dtype=torch.bfloat16,
        attn_implementation="flash_attention_2",
        device_map="auto"
    )
   
    model = PeftModel.from_pretrained(base_model, CONFIG['lora_path'])
    model.eval()
    print("✓ Model and adapter loaded successfully.")
   
    # ===========================
    # 3. Inference Loop
    # ===========================
    print(f"\n[4/4] Starting inference on {len(eval_df)} images...")
   
    predictions = []
    references = []
   
    for index, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Inference Progress"):
        image_path = os.path.join(CONFIG['image_dir'], row['filename'])
        ground_truth = str(row['roman']).strip()
       
        try:
            image = Image.open(image_path).convert("RGB")
        except Exception as e:
            continue
           
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": "识别满文单词："}, # Kept original prompt language to maintain model compatibility
                ],
            }
        ]
       
        text_prompt = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)
       
        inputs = processor(
            text=[text_prompt],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        ).to("cuda")
       
        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=30,
                do_sample=False,
                use_cache=True
            )
           
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
       
        pred_text = processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )[0].strip()
       
        predictions.append(pred_text)
        references.append(ground_truth)
   
    # ===========================
    # 4. Calculate Metrics (CER & WER)
    # ===========================
    print("\n" + "="*70)
    print("Evaluation complete. Calculating metrics...")
   
    clean_preds = [p if p else "<empty>" for p in predictions]
    clean_refs = [r if r else "<empty>" for r in references]
   
    cer = jiwer.cer(clean_refs, clean_preds)
    wer = jiwer.wer(clean_refs, clean_preds)
   
    print("\n[Evaluation Results (Unseen Words Mode - Qwen3-VL-4B QLoRA)]")
    print(f"Valid evaluation samples: {len(clean_refs)}")
    print(f"Character Error Rate (CER): {cer * 100:.2f}%")
    print(f"Word Error Rate (WER): {wer * 100:.2f}%")
   
    print("\n[Prediction Samples (Top 5)]")
    for i in range(min(5, len(clean_refs))):
        print(f"Ground truth: {clean_refs[i]}")
        print(f"Prediction:   {clean_preds[i]}")
        print("-" * 40)

if __name__ == "__main__":
    main()


libgomp: Invalid value for environment variable OMP_NUM_THREADS

libgomp: Invalid value for environment variable OMP_NUM_THREADS



Starting Qwen3-VL-4B QLoRA Manchu OCR Evaluation (Unseen Words Mode)

[1/4] Reproducing dataset split and checking overlap...
 > Total validation samples: 3000
 > Samples seen in training set (leakage): 1086
 > Label overlap rate: 36.20%

[!] Warning: Overlap rate (36.20%) exceeds threshold (5.0%).
[!] Filtering out images with seen labels...
 > Filtering complete. Final unseen validation set size: 1914

[2/4] Loading processor...

[3/4] Loading base model and integrating QLoRA adapter...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

/root/miniconda3/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


✓ Model and adapter loaded successfully.

[4/4] Starting inference on 1914 images...


Inference Progress: 100%|██████████| 1914/1914 [14:41<00:00,  2.17it/s]


Evaluation complete. Calculating metrics...

[Evaluation Results (Unseen Words Mode - Qwen3-VL-4B QLoRA)]
Valid evaluation samples: 1914
Character Error Rate (CER): 0.37%
Word Error Rate (WER): 2.61%

[Prediction Samples (Top 5)]
Ground truth: kurelembi
Prediction:   kurelembi
----------------------------------------
Ground truth: sonjome
Prediction:   sonjome
----------------------------------------
Ground truth: hederebumbihe
Prediction:   hederebumbihe
----------------------------------------
Ground truth: xeyehen
Prediction:   xeyehen
----------------------------------------
Ground truth: muserakvngge
Prediction:   muserakvngge
----------------------------------------
